In [15]:
from pathlib import Path
from collections import Counter
import optuna
from rectools.models import  PopularModel
from implicit.nearest_neighbours import CosineRecommender, BM25Recommender, TFIDFRecommender
from rectools.models.implicit_knn import ImplicitItemKNNWrapperModel
from rectools.dataset import Dataset
from rectools import Columns
from rectools.metrics import Precision, Recall, MAP, calc_metrics, MeanInvUserFreq, Serendipity
from rectools.model_selection import TimeRangeSplitter

import dill
import json
from pprint import pprint
import numpy as np
import scipy as sp
import time
import pandas as pd
import warnings

warnings.filterwarnings('ignore')


In [5]:
DATA_PATH = Path("../data/kion_train")

In [6]:
users = pd.read_csv(DATA_PATH / 'users.csv')
items = pd.read_csv(DATA_PATH / 'items.csv')
interactions = pd.read_csv(DATA_PATH / 'interactions.csv')


In [7]:
# rename columns, convert timestamp
interactions.rename(columns={'last_watch_dt': Columns.Datetime,
                            'total_dur': Columns.Weight},
                    inplace=True)

interactions['datetime'] = pd.to_datetime(interactions['datetime'])


In [8]:
interactions.head()


,user_id,item_id,datetime,weight,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


In [9]:
dataset = Dataset.construct(
    interactions_df=interactions,
    user_features_df=None,
    item_features_df=None
)


In [14]:
n_folds = 1
unit = "D"
n_units = 7
periods = n_folds + 1
freq = f"{n_units}{unit}"



# generator of folds
cv = TimeRangeSplitter(
    test_size=freq,
    n_splits=n_folds,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)


In [16]:
(train_ids, test_ids, fold_info) = next(cv.split(dataset.interactions, collect_fold_stats=True))


In [17]:
train_ids


array([      0,       1,       2, ..., 5476247, 5476249, 5476250])

In [18]:
train = interactions.loc[train_ids]
test = interactions.loc[test_ids]


In [19]:
test


,user_id,item_id,datetime,weight,watched_pct
9,203219,13582,2021-08-22,6975,100.0
54,200197,9335,2021-08-16,83,2.0
64,73446,14488,2021-08-19,6011,100.0
94,890735,14200,2021-08-16,1179,28.0
110,1053892,1916,2021-08-21,6502,100.0
...,...,...,...,...,...
5476159,1039219,3071,2021-08-22,3808,65.0
5476169,589589,983,2021-08-21,2403,43.0
5476188,590892,8618,2021-08-21,1335,23.0
5476191,857162,12360,2021-08-16,11,0.0


In [20]:
item_knn = ImplicitItemKNNWrapperModel(model=CosineRecommender(K=30))
item_knn.fit(dataset);


In [21]:
# take a look at the recommended items by the simple itemknn model
recs_itemknn = item_knn.recommend(
    test['user_id'].unique(),
    dataset=dataset,
    k=10,
    filter_viewed=False  # False - same items to every user
)


In [22]:
recs_itemknn.head()


,user_id,item_id,score,rank
0,203219,4976,8768.000001,1
1,203219,13865,7718.612431,2
2,203219,13582,6975.000056,3
3,203219,4880,5075.790326,4
4,203219,6936,3063.454010,5


## Обучение userKnn с различными мерами расстояния

In [23]:
max_date = interactions[Columns.Datetime].max()


In [24]:
interactions[Columns.Weight] = np.where(interactions['watched_pct'] > 10, 3, 1)


In [25]:
train = interactions[interactions[Columns.Datetime] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions[Columns.Datetime] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")


train: (4985269, 5)
test: (490982, 5)


In [26]:
cold_users = set(test[Columns.User]) - set(train[Columns.User])


In [27]:
test.drop(test[test[Columns.User].isin(cold_users)].index, inplace=True)


In [29]:
users.fillna('Unknown', inplace=True)

In [30]:
users = users.loc[users[Columns.User].isin(train[Columns.User])].copy()


In [31]:
users


,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0
5,1037719,age_45_54,income_60_90,М,0
...,...,...,...,...,...
840188,312839,age_65_inf,income_60_90,Ж,0
840189,191349,age_45_54,income_40_60,М,1
840190,393868,age_25_34,income_20_40,М,0
840192,339025,age_65_inf,income_0_20,Ж,0


In [32]:
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = users.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)
user_features = pd.concat(user_features_frames)
user_features.head()


,id,value,feature
0,973171,М,sex
1,962099,М,sex
3,721985,Ж,sex
4,704055,Ж,sex
5,1037719,М,sex


In [33]:
# оставляем у df users только тех, кто попал в train
items = items.loc[items[Columns.Item].isin(train[Columns.Item])].copy()


In [34]:
items["genre"] = items["genres"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
genre_feature = items[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature.head()


,id,value,feature
0,10711,драмы,genre
0,10711,зарубежные,genre
0,10711,детективы,genre
0,10711,мелодрамы,genre
1,2508,зарубежные,genre


In [35]:
content_feature = items.reindex(columns=[Columns.Item, "content_type"])
content_feature.columns = ["id", "value"]
content_feature["feature"] = "content_type"
content_feature.head()


,id,value,feature
0,10711,film,content_type
1,2508,film,content_type
2,10716,film,content_type
3,7868,film,content_type
4,16268,film,content_type


In [36]:
items["director"] = items["directors"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
director_feature = items[["item_id", "director"]].explode("director")
director_feature.columns = ["id", "value"]
director_feature["feature"] = "director"
director_feature.head()


,id,value,feature
0,10711,педро альмодовар,director
1,2508,скот армстронг,director
2,10716,адам п. калтраро,director
3,7868,эндрю хэй,director
4,16268,виктор садовский,director


In [37]:
items["country"] = items["countries"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
country_feature = items[["item_id", "country"]].explode("country")
country_feature.columns = ["id", "value"]
country_feature["feature"] = "country"
country_feature.head()


,id,value,feature
0,10711,испания,country
1,2508,сша,country
2,10716,канада,country
3,7868,великобритания,country
4,16268,ссср,country


In [38]:
year_feature = items.reindex(columns=[Columns.Item, "release_year"])
year_feature.columns = ["id", "value"]
year_feature["feature"] = "release_year"
year_feature.head()


,id,value,feature
0,10711,2002.0,release_year
1,2508,2014.0,release_year
2,10716,2011.0,release_year
3,7868,2015.0,release_year
4,16268,1978.0,release_year


In [39]:
# Объединяем фичи
item_features = pd.concat((genre_feature, content_feature, country_feature, year_feature, director_feature))
item_features


,id,value,feature
0,10711,драмы,genre
0,10711,зарубежные,genre
0,10711,детективы,genre
0,10711,мелодрамы,genre
1,2508,зарубежные,genre
...,...,...,...
15960,10632,амир камдин,director
15960,10632,эрик эгер,director
15961,4538,марк о’коннор,director
15961,4538,конор макмахон,director


In [40]:
metrics_name = {
    'Precision': Precision,
    'Recall': Recall,
    'MAP': MAP,
}

metrics = {}
for metric_name, metric in metrics_name.items():
    for k in range(1, 11):
        metrics[f'{metric_name}@{k}'] = metric(k=k)


pprint(metrics)


{'MAP@1': MAP(k=1, divide_by_k=False),
 'MAP@10': MAP(k=10, divide_by_k=False),
 'MAP@2': MAP(k=2, divide_by_k=False),
 'MAP@3': MAP(k=3, divide_by_k=False),
 'MAP@4': MAP(k=4, divide_by_k=False),
 'MAP@5': MAP(k=5, divide_by_k=False),
 'MAP@6': MAP(k=6, divide_by_k=False),
 'MAP@7': MAP(k=7, divide_by_k=False),
 'MAP@8': MAP(k=8, divide_by_k=False),
 'MAP@9': MAP(k=9, divide_by_k=False),
 'Precision@1': Precision(k=1),
 'Precision@10': Precision(k=10),
 'Precision@2': Precision(k=2),
 'Precision@3': Precision(k=3),
 'Precision@4': Precision(k=4),
 'Precision@5': Precision(k=5),
 'Precision@6': Precision(k=6),
 'Precision@7': Precision(k=7),
 'Precision@8': Precision(k=8),
 'Precision@9': Precision(k=9),
 'Recall@1': Recall(k=1),
 'Recall@10': Recall(k=10),
 'Recall@2': Recall(k=2),
 'Recall@3': Recall(k=3),
 'Recall@4': Recall(k=4),
 'Recall@5': Recall(k=5),
 'Recall@6': Recall(k=6),
 'Recall@7': Recall(k=7),
 'Recall@8': Recall(k=8),
 'Recall@9': Recall(k=9)}


Обучение

In [41]:
dataset = Dataset.construct(
    interactions_df=train,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type", "director", "country", "release_year"],
)
TEST_USERS = test[Columns.User].unique()


In [42]:
dataset.interactions.df


,user_id,item_id,weight,datetime
0,0,0,3.0,2021-05-11
1,1,1,3.0,2021-05-29
2,2,2,1.0,2021-05-09
3,3,3,3.0,2021-07-05
4,4,0,3.0,2021-04-30
...,...,...,...,...
5476244,69627,219,3.0,2021-08-02
5476245,40052,132,1.0,2021-05-12
5476246,896790,318,1.0,2021-08-13
5476247,206604,2546,3.0,2021-04-13


In [43]:
K_RECOS = 10

model = ImplicitItemKNNWrapperModel(model=BM25Recommender(K=100, K1=0.05, B=0.1, num_threads=2))
model.fit(dataset)
recos = model.recommend(
    users=TEST_USERS,
    dataset=dataset,
    k=K_RECOS,
    filter_viewed=True,
)


In [45]:
model.recommend(
    users=[123],
    dataset=dataset,
    k=K_RECOS,
    filter_viewed=True,
)


,user_id,item_id,score,rank
0,123,7571,25662.054643,1
1,123,7582,24845.530464,2
2,123,16166,23719.672696,3
3,123,1105,21246.525855,4
4,123,10761,20835.562336,5
5,123,3182,19528.632211,6
6,123,9506,18000.015473,7
7,123,11756,17670.804086,8
8,123,5411,16507.992780,9
9,123,13018,16034.600122,10


In [46]:
results = []
model_quality = {'model': 'BM25Recommender'}
metric_values = calc_metrics(metrics, recos, test, train)
model_quality.update(metric_values)
results.append(model_quality)

df_quality = pd.DataFrame(results).T

df_quality.columns = df_quality.iloc[0]

df_quality.drop('model', inplace=True)


In [51]:
dill_file = Path().cwd().parent / 'data' / 'weights'

with open(dill_file / 'BM25Recommender_0.095432.dill', 'wb') as f:
    dill.dump(model, f)


In [53]:
dill_file = Path().cwd().parent / 'data'

with open(dill_file / 'dataset_BM25Recommender_0.095432.dill', 'wb') as f:
    dill.dump(dataset, f)


Подбор гиперпараметров

In [54]:
dataset = Dataset.construct(
    interactions_df=train,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type", "director", "country", "release_year"],
)
TEST_USERS = test[Columns.User].unique()

results_opto = []

def objective(trial):
    # общие параметры
    K_RECOS = 10
    RANDOM_STATE = 42

    reco_model = trial.suggest_categorical("reco_model", ["BM25Recommender",
                                                          "CosineRecommender",
                                                          "TFIDFRecommender"])
    model_quality_opto = {"model": f"{reco_model}_{trial.number}"}

    if reco_model == "BM25Recommender":
        # гиперпараметры для BM25Recommender
        K = trial.suggest_int("K", 100, 500, 50, log=False)
        K1 = trial.suggest_float("K1", 0.01, 0.09, log=False)
        B = trial.suggest_float("B", 0.01, 0.5, log=False)
        # Инициализация BM25Recommender
        model = ImplicitItemKNNWrapperModel(model=BM25Recommender(K=K, K1=K1, B=B, num_threads=2))

    elif reco_model == "CosineRecommender":
        # гиперпараметры для CosineRecommender
        K = trial.suggest_int("K", 50, 200, 50, log=False)
        # Инициализация CosineRecommender
        model = ImplicitItemKNNWrapperModel(model=CosineRecommender(K=K))

    elif reco_model == "TFIDFRecommender":
        # гиперпараметры для TFIDFRecommender
        K = trial.suggest_int("K", 10, 100, 20, log=False)
        # Инициализация TFIDFRecommender
        model = ImplicitItemKNNWrapperModel(model=TFIDFRecommender(K=K))

    # обучение модели
    model.fit(dataset)
    recos = model.recommend(
        users=TEST_USERS,
        dataset=dataset,
        k=K_RECOS,
        filter_viewed=True,
    )

    # Подсчет метрик
    metric_values = calc_metrics(metrics, recos, test, train)
    model_quality_opto.update(metric_values)
    results_opto.append(model_quality_opto)

    return metric_values.get('MAP@10') # максимизируемая метрика


In [55]:
# запуск подбора гиперпараметров
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

pprint(f"Number of finished trials: {len(study.trials)}")
trial = study.best_trial
pprint(f"Best trial: {trial}")


[I 2023-11-29 15:47:29,448] A new study created in memory with name: no-name-89e4cc15-5562-46b9-89be-8d72908dc58a
[I 2023-11-29 15:47:42,409] Trial 0 finished with value: 0.07926813878289789 and parameters: {'reco_model': 'CosineRecommender', 'K': 100}. Best is trial 0 with value: 0.07926813878289789.
[I 2023-11-29 15:47:54,833] Trial 1 finished with value: 0.08046764264771092 and parameters: {'reco_model': 'TFIDFRecommender', 'K': 70}. Best is trial 1 with value: 0.08046764264771092.
[I 2023-11-29 15:48:08,315] Trial 2 finished with value: 0.08068690081724368 and parameters: {'reco_model': 'TFIDFRecommender', 'K': 90}. Best is trial 2 with value: 0.08068690081724368.
[I 2023-11-29 15:48:21,345] Trial 3 finished with value: 0.08068690081724368 and parameters: {'reco_model': 'TFIDFRecommender', 'K': 90}. Best is trial 2 with value: 0.08068690081724368.
[I 2023-11-29 15:48:34,859] Trial 4 finished with value: 0.07948747637554937 and parameters: {'reco_model': 'CosineRecommender', 'K': 15

'Number of finished trials: 100'
('Best trial: FrozenTrial(number=55, state=TrialState.COMPLETE, '
 'values=[0.08859721092713521], datetime_start=datetime.datetime(2023, 11, 29, '
 '15, 59, 56, 965689), datetime_complete=datetime.datetime(2023, 11, 29, 16, '
 "0, 11, 427138), params={'reco_model': 'BM25Recommender', 'K': 500, 'K1': "
 "0.07663166468644847, 'B': 0.013984294112838504}, user_attrs={}, "
 "system_attrs={}, intermediate_values={}, distributions={'reco_model': "
 "CategoricalDistribution(choices=('BM25Recommender', 'CosineRecommender', "
 "'TFIDFRecommender')), 'K': IntDistribution(high=500, log=False, low=100, "
 "step=50), 'K1': FloatDistribution(high=0.09, log=False, low=0.01, "
 "step=None), 'B': FloatDistribution(high=0.5, log=False, low=0.01, "
 'step=None)}, trial_id=55, value=None)')


In [56]:
df_quality = pd.DataFrame(results_opto).T

df_quality.columns = df_quality.iloc[0]

df_quality.drop('model', inplace=True)


In [57]:
df_quality = df_quality.T.drop_duplicates().T
df_quality.style.highlight_max(color='lightgreen', axis=1)


model,CosineRecommender_0,TFIDFRecommender_1,TFIDFRecommender_2,CosineRecommender_4,BM25Recommender_5,CosineRecommender_6,BM25Recommender_9,BM25Recommender_10,BM25Recommender_11,BM25Recommender_12,BM25Recommender_13,BM25Recommender_14,BM25Recommender_15,BM25Recommender_16,BM25Recommender_17,BM25Recommender_18,BM25Recommender_19,BM25Recommender_20,BM25Recommender_21,BM25Recommender_22,BM25Recommender_23,BM25Recommender_24,BM25Recommender_25,BM25Recommender_26,BM25Recommender_27,TFIDFRecommender_28,BM25Recommender_31,BM25Recommender_32,BM25Recommender_33,TFIDFRecommender_34,BM25Recommender_35,BM25Recommender_36,CosineRecommender_38,BM25Recommender_39,BM25Recommender_40,BM25Recommender_41,BM25Recommender_42,BM25Recommender_43,BM25Recommender_44,BM25Recommender_46,BM25Recommender_48,BM25Recommender_49,BM25Recommender_50,BM25Recommender_51,BM25Recommender_52,BM25Recommender_53,BM25Recommender_54,BM25Recommender_55,BM25Recommender_56,BM25Recommender_57,BM25Recommender_58,TFIDFRecommender_60,BM25Recommender_61,BM25Recommender_62,BM25Recommender_63,BM25Recommender_64,BM25Recommender_65,BM25Recommender_66,BM25Recommender_67,BM25Recommender_68,BM25Recommender_69,BM25Recommender_70,BM25Recommender_71,BM25Recommender_72,BM25Recommender_73,BM25Recommender_74,BM25Recommender_75,BM25Recommender_78,BM25Recommender_79,BM25Recommender_80,BM25Recommender_81,BM25Recommender_82,BM25Recommender_83,BM25Recommender_84,BM25Recommender_85,BM25Recommender_86,BM25Recommender_87,BM25Recommender_88,BM25Recommender_91,BM25Recommender_92,BM25Recommender_93,BM25Recommender_94,BM25Recommender_95,BM25Recommender_96,BM25Recommender_97,BM25Recommender_98,BM25Recommender_99
Precision@1,0.082850,0.083315,0.083364,0.083257,0.090592,0.082435,0.090741,0.091015,0.091131,0.090774,0.090957,0.085422,0.090733,0.084750,0.056431,0.089878,0.090948,0.087845,0.091156,0.089662,0.090608,0.090816,0.087090,0.090575,0.086069,0.081937,0.090757,0.090799,0.090948,0.079473,0.089853,0.090592,0.083315,0.088567,0.090799,0.090948,0.090990,0.090484,0.090791,0.090699,0.088542,0.090932,0.087862,0.090716,0.090965,0.090210,0.089961,0.091031,0.090915,0.090865,0.089812,0.083165,0.090948,0.090932,0.090990,0.091073,0.090749,0.090782,0.089795,0.089853,0.090998,0.090351,0.090699,0.090923,0.091031,0.090641,0.090807,0.091073,0.090733,0.088326,0.091114,0.091123,0.090583,0.090749,0.091023,0.090608,0.090666,0.090915,0.090774,0.090782,0.090766,0.090899,0.090807,0.090691,0.091031,0.089845,0.090683
Recall@1,0.041550,0.041703,0.041859,0.041644,0.046377,0.041474,0.046470,0.047264,0.047419,0.047491,0.047186,0.043088,0.047478,0.042706,0.026192,0.045864,0.047478,0.044424,0.047437,0.045650,0.046470,0.047457,0.044034,0.046357,0.043350,0.041124,0.047440,0.046617,0.047499,0.040013,0.045792,0.046407,0.041663,0.044976,0.046869,0.047388,0.047384,0.047388,0.047035,0.046565,0.044960,0.047170,0.044454,0.047461,0.047186,0.047347,0.045918,0.047502,0.047187,0.047039,0.045785,0.041777,0.047492,0.047183,0.047236,0.047237,0.047420,0.046631,0.045776,0.045738,0.047239,0.047330,0.047450,0.047041,0.047219,0.046529,0.046682,0.047399,0.047490,0.044823,0.047399,0.047416,0.046392,0.047032,0.047379,0.046428,0.047042,0.047374,0.047491,0.047492,0.047443,0.047154,0.047502,0.047065,0.047310,0.045732,0.046508
Precision@2,0.069927,0.070408,0.070404,0.070026,0.076245,0.069180,0.076461,0.076664,0.076660,0.076465,0.076747,0.071673,0.076482,0.071387,0.054133,0.076096,0.076469,0.073603,0.076693,0.075602,0.076536,0.076453,0.073030,0.076287,0.072159,0.069022,0.076473,0.076673,0.076444,0.067417,0.076104,0.076436,0.070213,0.075196,0.076789,0.076652,0.076569,0.076428,0.076768,0.076581,0.075221,0.076768,0.074046,0.076486,0.076785,0.076353,0.076158,0.076494,0.076747,0.076822,0.076083,0.069910,0.076465,0.076776,0.076789,0.076760,0.076473,0.076673,0.076112,0.075793,0.076785,0.076374,0.076482,0.076768,0.076751,0.076573,0.076702,0.076697,0.076465,0.074669,0.076714,0.076706,0.076419,0.076768,0.076660,0.076287,0.076809,0.076644,0.076461,0.076461,0.076457,0.07684

Обучение модели с лучшими гиперпараметрами


In [58]:
users_inv_mapping = dict(enumerate(train['user_id'].unique()))
users_mapping = {v: k for k, v in users_inv_mapping.items()}


items_inv_mapping = dict(enumerate(train['item_id'].unique()))
items_mapping = {v: k for k, v in items_inv_mapping.items()}


In [59]:
def get_coo_matrix(df,
                   user_col='user_id',
                   item_col='item_id',
                   weight_col=None,
                   users_mapping=None,
                   items_mapping=None):
    if weight_col:
        weights = df[weight_col].astype(np.float32)
    else:
        weights = np.ones(len(df), dtype=np.float32)

    interaction_matrix = sp.sparse.coo_matrix((
        weights,
        (
            df[user_col].map(users_mapping.get),
            df[item_col].map(items_mapping.get)
        )
    ))
    return interaction_matrix
interaction_matrix = get_coo_matrix(train, weight_col='weight',
                                    users_mapping=users_mapping,
                                    items_mapping=items_mapping)

Обучение

In [61]:
userknn = BM25Recommender(K=50, K1=0.012556305101247701, B=0.05289835164246949, num_threads=2)
userknn.fit(interaction_matrix)


  0%|          | 0/15565 [00:00<?, ?it/s]

In [64]:
dill_file = Path().cwd().parent / 'data' / 'weights'

with open(dill_file / 'userknn_BM25Recommender.dill', 'wb') as f:
    dill.dump(userknn, f)


In [65]:
dill_file = Path().cwd().parent / 'data'

with open(dill_file / 'dataset_userknn_BM25Recommender.dill', 'wb') as f:
    dill.dump(dataset, f)
